# Requirements

In [6]:
import os, sys, platform
node = platform.node()
if "arctrd" in node:
    ROOT_DIR = os.getcwd()
else:
    ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(ROOT_DIR)

from omegaconf import OmegaConf, DictConfig
import numpy as np
import pandas as pd
import torch 
import os
import time

import importlib
import asset_scripts.scripts.plot_losses 

from src.settings import LOGS_ROOT, DATA_ROOT

from src.datasets.ukb_hold import load_data_hold as load_ukb_pretrain

UKB_DATADICT, demo_df = load_ukb_pretrain(
    file_path=os.path.join(DATA_ROOT, "ukb_ica/ukb_data_hold.npz"),
    demo_path=os.path.join(DATA_ROOT, "ukb_ica/demographics_legend_hold.csv"),
)

UKB_DATA = UKB_DATADICT['data']
print("Available data in ukb datadict:",UKB_DATADICT.keys())
print("UKB data shape:", UKB_DATA.shape)
# prepare train and validation sets
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(UKB_DATA, test_size=0.2, random_state=42)

from src.datasets.fbirn import load_data_hold as load_fbirn_hold

FBIRN_DATADICT, fbirn_demo_df = load_fbirn_hold()

FBIRN_DATA = FBIRN_DATADICT['data']
print("Available data in fbirn datadict:",FBIRN_DATADICT.keys())
print("FBIRN data shape:", FBIRN_DATA.shape)

from src.datasets.ukb_aal_hold import load_data_hold as load_aal_hold

AAL_DATA = load_aal_hold()
print("AAL UKB data shape:", AAL_DATA.shape)

Available data in ukb datadict: dict_keys(['data', 'sexes', 'ages', 'age_bins'])
UKB data shape: (1196, 490, 53)
Available data in fbirn datadict: dict_keys(['data', 'diags', 'sexes', 'ages', 'age_bins'])
FBIRN data shape: (16, 140, 53)
AAL UKB data shape: (100, 490, 166)


In [7]:
import importlib
import asset_scripts.scripts.plot_losses
from asset_scripts.scripts.plot_matrices import get_model_class, get_model_results, \
    load_dataset, derive_matrices_for_model, plot_combined_matrices, \
        plot_mean_matrices, plot_aggregated_matrices
importlib.reload(asset_scripts.scripts.plot_matrices)
from asset_scripts.scripts.plot_matrices import get_model_class, get_model_results, \
    load_dataset, derive_matrices_for_model, plot_combined_matrices, \
        plot_mean_matrices, plot_aggregated_matrices
        
def inspect_model(exp_path, test_data, use_cache=True, n_samples=5, null_diagonal=False):

    if not os.path.exists(exp_path):
        print(f"Error: Experiment path {exp_path} does not exist.")

    if null_diagonal:
        null_matrices_list = []
    matrices_list = []
    titles_list = []
    losses = []

    if n_samples == 0:
        n_samples = test_data.shape[0]


    for run in sorted(os.listdir(exp_path)):
        run_path = os.path.join(exp_path, run)
        if not os.path.isdir(run_path):
            continue

        print(f"Processing run {run}...")
    
        try:
            matrices, predicted, originals, loss, best_idx = get_model_results(run_path, test_data, use_cache=use_cache, n_samples=n_samples)
            if matrices is not None:
                if null_diagonal:
                    i = np.arange(53)
                    matrices = matrices.copy()
                    matrices[..., i, i] -= 1

                title = f"Run {run} | Best Epoch: {best_idx}"
                if loss is not None:
                    title += f" | Loss: {loss:.4f}"
                    losses.append(loss)
                else:
                    # Append infinity so runs without a valid loss are sorted to the very end
                    losses.append(float('inf')) 
                    
                matrices_list.append(matrices)
                titles_list.append(title)
                
                # Plot individually
                save_path = os.path.join(exp_path, f"_images/{run}.png")
                plot_combined_matrices(matrices, save_path, n_samples=n_samples)
                
                mean_save_path = os.path.join(exp_path, f"_images/{run}_mean.png")
                plot_mean_matrices(matrices, mean_save_path, n_samples=n_samples)

                # if null_diagonal:
                #     i = np.arange(53)
                #     matrices = matrices.copy()
                #     matrices[..., i, i] = 0
                #     null_matrices_list.append(matrices)


                #     save_path = os.path.join(exp_path, f"_images/{run}_null.png")
                #     plot_combined_matrices(matrices, save_path, n_samples=n_samples)
                    
                #     mean_save_path = os.path.join(exp_path, f"_images/{run}_null_mean.png")
                #     plot_mean_matrices(matrices, mean_save_path, n_samples=n_samples)
                    
            else:
                print(f"No valid results derived for run {run}.")
        except Exception as e:
            print(f"Error processing run {run}: {e}")

    # --- ADD THIS SORTING LOGIC HERE ---
    if matrices_list and losses:
        # Zip the lists together, sort by the first element (loss), and unzip
        sorted_combined = sorted(zip(losses, matrices_list, titles_list), key=lambda x: x[0])
        losses, matrices_list, titles_list = zip(*sorted_combined)
        
        # zip(*...) returns tuples, so convert them back to lists
        matrices_list = list(matrices_list)
        titles_list = list(titles_list)
    # -----------------------------------

    if matrices_list:
        aggregated_save_path = os.path.join(exp_path, f"_images/aggregated_matrices.png")
        plot_aggregated_matrices(matrices_list, titles_list, aggregated_save_path, n_samples=n_samples)
        print(f"Aggregated plot saved to {aggregated_save_path}")


In [9]:
# old
test_data = load_dataset("fbirn")
exp_path = "/Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb-DECIFRA"
use_cache=False
n_samples = 1

inspect_model(exp_path, test_data, use_cache=use_cache, n_samples=n_samples)


Processing run 00...
Processing run 01...
Processing run 02...
Processing run 03...
Processing run 04...
Processing run 05...
Processing run 06...
Processing run 07...
Processing run 08...
Processing run 09...
Processing run 10...
Processing run 11...
Processing run 12...
Processing run 13...
Processing run 14...
Processing run 15...
Processing run 16...
Processing run 17...
Processing run 18...
Processing run 19...
Processing run _images...
No valid results derived for run _images.
Aggregated plot saved to /Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb-DECIFRA/_images/aggregated_matrices.png


In [4]:
# default
test_data = load_dataset("fbirn")
exp_path = "/Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb-DECIFRA_MS-default"
use_cache=False
n_samples = 1

inspect_model(exp_path, test_data, use_cache=use_cache, n_samples=n_samples, null_diagonal=True)


Processing run 10...
Setting transfer weight to 0.1994269340974211
Processing run 11...
Setting transfer weight to 0.2
Processing run 12...
Setting transfer weight to 0.1977077363896847
Processing run 13...
Setting transfer weight to 0.1965616045845273
Processing run 14...
Setting transfer weight to 0.2
Processing run 15...
Setting transfer weight to 0.1977077363896847
Processing run 16...
Setting transfer weight to 0.1948424068767908
Processing run 17...
Setting transfer weight to 0.1988538681948424
Processing run 18...
Setting transfer weight to 0.1994269340974211
Processing run 19...
Setting transfer weight to 0.1948424068767908
Processing run _images...
Error processing run _images: not enough values to unpack (expected 5, got 3)
Processing run bs...
Error processing run bs: not enough values to unpack (expected 5, got 3)
Aggregated plot saved to /Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb-DECIFRA_MS-default/_images/aggregated_matrices.png


In [5]:
# default on half ukb
test_data = load_dataset("fbirn")
exp_path = "/Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb_half-DECIFRA_MS-default"
use_cache=False
n_samples = 1

inspect_model(exp_path, test_data, use_cache=use_cache, n_samples=n_samples, null_diagonal=True)


Processing run 00...
Setting transfer weight to 0.1982808022922637
Processing run 01...
Setting transfer weight to 0.2
Processing run 02...
Setting transfer weight to 0.1873925501432665
Processing run 03...
Setting transfer weight to 0.1977077363896847
Processing run 04...
Setting transfer weight to 0.197134670487106
Processing run 05...
Setting transfer weight to 0.1037249283667622
Processing run 06...
Setting transfer weight to 0.1524355300859599
Processing run 07...
Setting transfer weight to 0.1994269340974211
Processing run 08...
Setting transfer weight to 0.197134670487106
Processing run 09...
Setting transfer weight to 0.1982808022922637
Processing run 10...
Setting transfer weight to 0.1988538681948424
Processing run 11...
Setting transfer weight to 0.197134670487106
Processing run 12...
Setting transfer weight to 0.1994269340974211
Processing run 13...
Setting transfer weight to 0.1982808022922637
Processing run 14...
Setting transfer weight to 0.2
Processing run 15...
Setting

In [6]:
# default on ukb+1000
test_data = load_dataset("fbirn")
exp_path = "/Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb_plus_1000-DECIFRA_MS-default"
use_cache=False
n_samples = 1

inspect_model(exp_path, test_data, use_cache=use_cache, n_samples=n_samples, null_diagonal=True)


Processing run 00...
Setting transfer weight to 0.1914040114613182
Processing run 01...
Setting transfer weight to 0.1982808022922635
Processing run 02...
Setting transfer weight to 0.1851002865329513
Processing run 03...
Setting transfer weight to 0.1994269340974213
Processing run 04...
Setting transfer weight to 0.1931232091690544
Processing run 05...
Setting transfer weight to 0.1977077363896847
Processing run 06...
Setting transfer weight to 0.1931232091690544
Processing run 07...
Setting transfer weight to 0.2000000000000001
Processing run 08...
Setting transfer weight to 0.197134670487106
Processing run 09...
Setting transfer weight to 0.1965616045845273
Processing run 10...
Setting transfer weight to 0.197134670487106
Processing run 11...
Setting transfer weight to 0.1982808022922635
Processing run 12...
Setting transfer weight to 0.1977077363896847
Processing run 13...
Setting transfer weight to 0.1965616045845273
Processing run 14...
Setting transfer weight to 0.19369627507163

In [ ]:
# default UKB for exp_path = "/Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb-DECIFRA_MS-default" run 14
test_data = load_dataset("ukb")
run_path = "/Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb-DECIFRA_MS-default/14"
use_cache=False
n_samples = 0

# test_data = test_data[:600]
test_data = test_data[:600]
matrices, predicted, originals, loss, best_idx = get_model_results(run_path, test_data, use_cache=use_cache, n_samples=n_samples)
test_data = test_data[600:]
matrices_2, predicted, originals, loss, best_idx = get_model_results(run_path, test_data, use_cache=use_cache, n_samples=n_samples)

np.save("/Users/ppopov1/DECIFRA/scripts/asset_scripts/ukb_matrices_1.npy", matrices)
np.save("/Users/ppopov1/DECIFRA/scripts/asset_scripts/ukb_matrices_2.npy", matrices_2)
matrices_all = np.concatenate((matrices, matrices_2))
matrices_all.shape
np.save("/Users/ppopov1/DECIFRA/scripts/asset_scripts/ukb_matrices.npy", matrices_all)


Setting transfer weight to 0.2


In [43]:
# no-mix
test_data = load_dataset("fbirn")
exp_path = "/Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb-DECIFRA_MS-noMixed"
use_cache=True
n_samples = 1

inspect_model(exp_path, test_data, use_cache=use_cache, n_samples=n_samples)


Processing run 00...
Processing run 01...
Processing run 02...
Processing run 03...
Processing run 04...
Processing run 05...
Processing run 06...
Processing run 07...
Processing run 08...
Processing run 09...
Processing run 10...
No valid results derived for run 10.
Processing run 11...
No valid results derived for run 11.
Processing run _images...
No valid results derived for run _images.
Aggregated plot saved to /Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb-DECIFRA_MS-noMixed/_images/aggregated_matrices.png


In [7]:
# inspect separate checkpoints

test_data = load_dataset("fbirn")
exp_path = "/Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb-DECIFRA_MS-default/02/"
use_cache=False
n_samples = 1

matrices, losses, best_idxs, titles = [], [], [], []
epochs = [0, 30, 60, 90, 110, 130, 200, 400, 499]
for epoch in epochs:
    matrix, loss, best_idx = get_model_results(exp_path, test_data, checkpoint=epoch, use_cache=False, n_samples=n_samples)
    matrices.append(matrix)
    losses.append(loss)
    best_idxs.append(best_idx)

    title = f"Epoch {epoch:04d} | Loss: {loss:.4f}"
    titles.append(title)
    


In [8]:

aggregated_save_path = "/Users/ppopov1/DECIFRA/scripts/assets/plots/multistage_inscpect.png"
plot_aggregated_matrices(matrices, titles, aggregated_save_path, n_samples=n_samples)
print(f"Aggregated plot saved to {aggregated_save_path}")

Aggregated plot saved to /Users/ppopov1/DECIFRA/scripts/assets/plots/multistage_inscpect.png


## Fingerprinting

In [6]:
from asset_scripts.scripts.fingerprinting import compute_cohens_d_one_vs_rest, compute_mahalanobis_one_vs_rest, train_ml_fingerprinter, transform_to_domains
importlib.reload(asset_scripts.scripts.fingerprinting)
from asset_scripts.scripts.fingerprinting import compute_cohens_d_one_vs_rest, compute_mahalanobis_one_vs_rest, train_ml_fingerprinter, transform_to_domains

ukb_matrices = np.load("/Users/ppopov1/DECIFRA/scripts/asset_scripts/ukb_matrices.npy")
ukb_matrices.shape
# remove the diagonal
i = np.arange(53)
ukb_matrices[..., i, i] -= 1

In [ ]:
mah_d = compute_mahalanobis_one_vs_rest(ukb_matrices)

mean, var = np.mean(mah_d, axis=0), np.std(mah_d, axis=0)
print(mean, var)

In [7]:
coh_d = compute_cohens_d_one_vs_rest(ukb_matrices)
coh_d = np.abs(coh_d)

mean, var = np.mean(coh_d, axis=0), np.std(coh_d, axis=0)
# print(mean, var)
mmean, vvar = np.mean(mean), np.std(var)
print(mmean, vvar)

2026-03-30 19:07:53,066 - INFO - Computing Cohen's d for data of shape (1196, 490, 53, 53)
2026-03-30 19:07:53,068 - INFO - Pre-computing per-subject means and variances...
2026-03-30 19:07:55,331 - INFO - Calculating one-vs-rest effect sizes...
2026-03-30 19:07:55,617 - INFO -   Processed 100/1196 subjects
2026-03-30 19:07:55,950 - INFO -   Processed 200/1196 subjects
2026-03-30 19:07:56,221 - INFO -   Processed 300/1196 subjects
2026-03-30 19:07:56,524 - INFO -   Processed 400/1196 subjects
2026-03-30 19:07:56,782 - INFO -   Processed 500/1196 subjects
2026-03-30 19:07:57,045 - INFO -   Processed 600/1196 subjects
2026-03-30 19:07:57,302 - INFO -   Processed 700/1196 subjects
2026-03-30 19:07:57,559 - INFO -   Processed 800/1196 subjects
2026-03-30 19:07:57,815 - INFO -   Processed 900/1196 subjects
2026-03-30 19:07:58,070 - INFO -   Processed 1000/1196 subjects
2026-03-30 19:07:58,327 - INFO -   Processed 1100/1196 subjects
2026-03-30 19:07:58,576 - INFO - Cohen's d calculation took

0.057212017 0.007269869


In [8]:
cropped_data = transform_to_domains(ukb_matrices)
cropped_data.shape

2026-03-30 19:07:59,841 - INFO - Loading domain mapping from /Users/ppopov1/DECIFRA/assets/data/ICN_coordinates.csv
2026-03-30 19:07:59,852 - INFO - Found 7 domains: ['Subcortical', 'Auditory', 'Sensorimotor', 'Visual', 'Cognitive Control', 'Default mode', 'Cerebellum']
2026-03-30 19:07:59,862 - INFO - Transforming data from 53x53 to 7x7 by averaging cross-domain ranges...
2026-03-30 19:08:04,874 - INFO - Transformation complete. Condensed shape is now: (1196, 490, 7, 7)


(1196, 490, 7, 7)

In [9]:
aucs = train_ml_fingerprinter(cropped_data)

2026-03-30 19:08:09,746 - INFO - Preparing ML fingerprinting experiment...
2026-03-30 19:08:09,747 - INFO - Flattening samples: 586040 total instances, 49 features
2026-03-30 19:08:09,748 - INFO - Splitting data into train and test sets (test_size=0.2)...
2026-03-30 19:08:09,906 - INFO - Using SGDClassifier with log_loss as the base Logistic Regression...
2026-03-30 19:08:09,907 - INFO - Training single multiclass classifier on training set...
2026-03-30 19:09:54,766 - INFO - Model training completed in 104.86 seconds
2026-03-30 19:09:54,767 - INFO - Evaluating predictions on holdout set...
2026-03-30 19:10:07,461 - INFO - ====== Evaluation Metrics ======
2026-03-30 19:10:07,461 - INFO -   Accuracy:      0.0008
2026-03-30 19:10:07,461 - INFO -   Avg Precision: 0.0000
2026-03-30 19:10:07,461 - INFO -   Avg Recall:    0.0008
2026-03-30 19:10:07,462 - INFO -   Avg AUC (OvR): 0.5320
2026-03-30 19:10:07,462 - INFO - ================================


In [10]:
coh_d = compute_cohens_d_one_vs_rest(cropped_data)
coh_d = np.abs(coh_d)

mean, var = np.mean(coh_d, axis=0), np.std(coh_d, axis=0)
# print(mean, var)
mmean, vvar = np.mean(mean), np.std(var)
print(mmean, vvar)

2026-03-30 19:13:30,175 - INFO - Computing Cohen's d for data of shape (1196, 490, 7, 7)
2026-03-30 19:13:30,176 - INFO - Pre-computing per-subject means and variances...
2026-03-30 19:13:30,236 - INFO - Calculating one-vs-rest effect sizes...
2026-03-30 19:13:30,250 - INFO -   Processed 100/1196 subjects
2026-03-30 19:13:30,266 - INFO -   Processed 200/1196 subjects
2026-03-30 19:13:30,281 - INFO -   Processed 300/1196 subjects
2026-03-30 19:13:30,295 - INFO -   Processed 400/1196 subjects
2026-03-30 19:13:30,308 - INFO -   Processed 500/1196 subjects
2026-03-30 19:13:30,323 - INFO -   Processed 600/1196 subjects
2026-03-30 19:13:30,337 - INFO -   Processed 700/1196 subjects
2026-03-30 19:13:30,354 - INFO -   Processed 800/1196 subjects
2026-03-30 19:13:30,371 - INFO -   Processed 900/1196 subjects
2026-03-30 19:13:30,386 - INFO -   Processed 1000/1196 subjects
2026-03-30 19:13:30,400 - INFO -   Processed 1100/1196 subjects
2026-03-30 19:13:30,416 - INFO - Cohen's d calculation took 0

0.069940515 0.0094390595


In [11]:
mah_d = compute_mahalanobis_one_vs_rest(cropped_data)

mean, var = np.mean(mah_d, axis=0), np.std(mah_d, axis=0)
print(mean, var)

2026-03-30 19:14:12,761 - INFO - Computing Mahalanobis Distances for data of shape (1196, 490, 7, 7)
2026-03-30 19:14:12,762 - INFO - Computing global uncentered scatter matrix (may take a minute)...
2026-03-30 19:14:12,928 - INFO - Calculating one-vs-rest Mahalanobis distances...
2026-03-30 19:14:12,936 - INFO -   Processed 50/1196 subjects
2026-03-30 19:14:12,943 - INFO -   Processed 100/1196 subjects
2026-03-30 19:14:12,952 - INFO -   Processed 150/1196 subjects
2026-03-30 19:14:12,959 - INFO -   Processed 200/1196 subjects
2026-03-30 19:14:12,968 - INFO -   Processed 250/1196 subjects
2026-03-30 19:14:12,976 - INFO -   Processed 300/1196 subjects
2026-03-30 19:14:12,984 - INFO -   Processed 350/1196 subjects
2026-03-30 19:14:12,992 - INFO -   Processed 400/1196 subjects
2026-03-30 19:14:13,000 - INFO -   Processed 450/1196 subjects
2026-03-30 19:14:13,008 - INFO -   Processed 500/1196 subjects
2026-03-30 19:14:13,016 - INFO -   Processed 550/1196 subjects
2026-03-30 19:14:13,023 - 

0.6207102 0.13052797
